# Condition A , XGBoost with Median Imputation

Training XGBoost on Strategy A: median imputation with 48 lag features across 8 vital signs (88 features total). This is the baseline tree-based condition.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import joblib
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from src.config import RANDOM_SEED, DATA_DIR, MODELS_DIR
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers
from src.features import add_lag_features
from src.utils import set_all_seeds, validate_no_nans, create_patient_splits
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
print('Imports OK')

## 1. Load Data and Create Splits

Loading the full dataset and applying the stratified 70/15/15 patient-level split. Stratification preserves the 5.62% sepsis prevalence across train, validation, and test sets.

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Train: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}')

## 2. Lag Feature Engineering

Adding 48 temporal lag features across the 8 vital signs , computed after the split to prevent data leakage. These give XGBoost explicit access to trend information (rate of change, rolling averages) that it could not otherwise derive from a single-timestep snapshot.

In [ ]:
# Per-patient groupby — no cross-patient leakage
train_df = add_lag_features(train_df)
val_df   = add_lag_features(val_df)
test_df  = add_lag_features(test_df)
print(f'Columns after lag features: {len(train_df.columns)}')

## 3. Strategy A Preprocessing

Applying median imputation using per-feature training medians, followed by standard scaling. Imputation parameters are fitted on the training set only and applied to validation and test sets.

In [ ]:
meta_cols    = {'patient_id', 'hospital_id', 'SepsisLabel', 'EarlyLabel', 'timestep'}
feature_cols = [c for c in train_df.columns if c not in meta_cols]

X_train_raw = train_df[feature_cols].values
X_val_raw   = val_df[feature_cols].values
X_test_raw  = test_df[feature_cols].values

y_train = train_df['EarlyLabel'].values
y_val   = val_df['EarlyLabel'].values
y_test  = test_df['EarlyLabel'].values

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train_raw)
X_val   = imputer.transform(X_val_raw)
X_test  = imputer.transform(X_test_raw)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

for arr, name in [(X_train,'train_A'), (X_val,'val_A'), (X_test,'test_A')]:
    validate_no_nans(arr, name, feature_cols)

os.makedirs(MODELS_DIR, exist_ok=True)
joblib.dump(imputer,      f'{MODELS_DIR}strategy_A_lag_imputer.pkl')
joblib.dump(scaler,       f'{MODELS_DIR}strategy_A_lag_scaler.pkl')
joblib.dump(feature_cols, f'{MODELS_DIR}strategy_A_lag_feature_names.pkl')
print(f'Features: {X_train.shape[1]} | Train rows: {X_train.shape[0]:,} | mean ≈ {X_train.mean():.4f}')

## 4. XGBoost Hyperparameter Search

Grid search over tree depth ∈ {3, 4, 6} and learning rate ∈ {0.05, 0.1}, with 500 estimators and early stopping on validation AUPRC (patience = 20). Class imbalance is handled via `scale_pos_weight = 43.3`.

In [ ]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f'scale_pos_weight = {pos_weight:.1f}')

param_grid = [{'max_depth': d, 'learning_rate': lr}
              for d in [3, 4, 6] for lr in [0.05, 0.1]]

best_val_auprc, best_params, best_model = 0.0, None, None

for params in param_grid:
    m = xgb.XGBClassifier(
        n_estimators=500, max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        scale_pos_weight=pos_weight, eval_metric='aucpr',
        early_stopping_rounds=20, random_state=RANDOM_SEED, n_jobs=-1,
    )
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_auprc = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
    print(f"depth={params['max_depth']} lr={params['learning_rate']} val AUPRC={val_auprc:.4f}")
    if val_auprc > best_val_auprc:
        best_val_auprc, best_params, best_model = val_auprc, params, m

print(f'Best: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 5. Evaluate and Log

Evaluating the best model on the held-out test set and appending results to `experiment_log.csv`. Reporting AUPRC, AUC-ROC, F1, precision, and recall at the threshold that maximised F1 on the validation set.

In [ ]:
val_probs  = best_model.predict_proba(X_val)[:, 1]
test_probs = best_model.predict_proba(X_test)[:, 1]

threshold    = select_threshold(y_val, val_probs)
val_metrics  = compute_all_metrics(y_val,  val_probs,  threshold)
test_metrics = compute_all_metrics(y_test, test_probs, threshold)

print(f'Test AUC-ROC: {test_metrics["auc_roc"]:.4f} | AUPRC: {test_metrics["auprc"]:.4f} | F1: {test_metrics["f1"]:.4f}')

log_results('A', 'XGBoost', 'Strategy_A', val_metrics, test_metrics, best_params)

joblib.dump(best_model, f'{MODELS_DIR}xgb_condition_A.pkl')
print(f'Saved | Features: {best_model.n_features_in_}')